# PRACTICE FILE — Encoding & Scikit-learn Pipelines
### Dataset: IBM Telco Customer Churn

**Rules of this notebook:**
- Every section has a **short brief** (3-5 lines) reminding you of the concept. That is NOT the lesson — you already have the full guide (`Encoding_Pipelines_Guide.docx`) and the worked example (`Telco_Churn_Encoding_Pipelines.ipynb`). Go back to those if a brief isn't enough.
- Every brief is followed by **Tasks**. You write the code in the empty cell below each task. Cells marked `# TODO` are yours to complete — do not skip them, do not peek at the worked-example notebook while attempting a task for the first time.
- Tasks marked **(Challenge)** are harder and optional-but-expected if you want to actually be good at this.
- Some cells contain a **Self-check** — run it after your task. If it prints an error, your code is wrong. Fix it before moving on. A self-check passing does not guarantee a perfect answer, but a failing one guarantees a broken one.
- No solutions are provided in this file. That is the point of a practice file.

**Setup below is given to you — do not skip running it.**


## 0. Setup (Given — just run these)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

print("Setup complete.")

Setup complete.


In [10]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)
df = df.drop(columns=['customerID'])

print("Shape:", df.shape)
df.head()

Shape: (7043, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 


## 1. Encoding — Brief

Machine learning models only understand numbers. Encoding converts
categorical (text) columns into numeric form. **Nominal** data (no
order, e.g. `gender`) usually gets **One-Hot Encoding**. **Ordinal**
data (real order, e.g. `Contract`) gets **Ordinal Encoding** with a
manually defined order. High-cardinality columns often use
**Frequency** or **Target Encoding**. Label Encoding is best kept for
the target column or tree-based models.


### Task 1.1 — Clean `TotalCharges`
`TotalCharges` is loaded as text (`object`) because a few rows contain
blank strings instead of numbers.
- Convert `TotalCharges` to a numeric dtype.
- Print how many missing values exist after the conversion.
- Do **not** drop or fill them yet — that comes later with a proper `SimpleImputer`.


In [ ]:
# TODO: convert df['TotalCharges'] to numeric, coercing errors to NaN
# TODO: print the number of missing values in TotalCharges after conversion

df['TotalCharges']= pd.to_numeric(df['TotalCharges'], errors='coerce')
print( "Missing values in TotalCharges now",df['TotalCharges'].isna().sum()) #isna() tells missing values

Missing values in TotalCharges now 11


### Task 1.2 — Label Encode the target column
Encode `Churn` ('Yes'/'No') into a numeric column called `Churn_encoded`
using `LabelEncoder`. Print which string mapped to 0 and which to 1.


In [38]:
le = LabelEncoder()
df['Churn_encoded'] = le.fit_transform(df['Churn'])
df.head(5)
# print(dict(zip(le.classes_, le.transform(le.classes_))))

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Churn_encoded
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1


### Task 1.3 — Train/Test split BEFORE further encoding
Split the data into `X_train`, `X_test`, `y_train`, `y_test`
(80/20, `random_state=42`, `stratify` on the target). Use `Churn_encoded`
as `y`, and drop both `Churn` and `Churn_encoded` from `X`.

**Why does the split have to happen before you fit any Frequency/Target
encoder?** Answer this in a one-line comment in your code cell.


In [39]:
X = df.drop(columns=['Churn', 'Churn_encoded'])
y = df['Churn_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# Splitting first keeps the test set completely unseen, so an encoder that looks at y (like target encoding) can't leak test information into training.

### Self-check 1.3

In [14]:
# Self-check — do not edit, just run
try:
    assert X_train.shape[0] == 5634 and X_test.shape[0] == 1409
    assert 'Churn' not in X_train.columns and 'Churn_encoded' not in X_train.columns
    print("Self-check passed.")
except NameError:
    print("X_train / X_test not defined yet — complete Task 1.3 first.")
except AssertionError:
    print("Shapes or columns look wrong — check your split and dropped columns.")

Self-check passed.


### Task 1.4 — One-Hot Encode two nominal columns
Using `OneHotEncoder` (with `drop='first'` and `handle_unknown='ignore'`),
one-hot encode the `gender` and `InternetService` columns of `X_train`.
Print the resulting shape and the generated column names
(`get_feature_names_out`).


In [40]:
ohe = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
encoded = ohe.fit_transform(X_train[['gender', 'InternetService']])
print(encoded.shape)
print(ohe.get_feature_names_out(['gender', 'InternetService']))

(5634, 3)
['gender_Male' 'InternetService_Fiber optic' 'InternetService_No']


### Task 1.5 — Ordinal Encode `Contract`
`Contract` has a real order: `Month-to-month` < `One year` < `Two year`.
Use `OrdinalEncoder` with a manually specified `categories` order (do
**not** let it guess alphabetically) to create `Contract_ordinal`.
Print the unique (`Contract`, `Contract_ordinal`) pairs sorted by the
encoded value, to prove the order is correct.


In [16]:
oe = OrdinalEncoder(categories=[['Month-to-month', 'One year', 'Two year']])
X_train_ord = X_train.copy()
X_train_ord['Contract_ordinal'] = oe.fit_transform(X_train_ord[['Contract']])
print(X_train_ord[['Contract', 'Contract_ordinal']].drop_duplicates().sort_values('Contract_ordinal'))

            Contract  Contract_ordinal
3738  Month-to-month               0.0
2257        One year               1.0
4860        Two year               2.0


### Task 1.6 — Frequency Encode `PaymentMethod`
Implement Frequency Encoding **manually with pandas** (no sklearn class
for this one). Use only `X_train` to compute the frequency map — this
matters, think about why. Apply the same map to a copy of `X_train`
as a new column `PaymentMethod_freq`.


In [43]:
print("paymentmethod", df['PaymentMethod'])

paymentmethod 0                Electronic check
1                    Mailed check
2                    Mailed check
3       Bank transfer (automatic)
4                Electronic check
                  ...            
7038                 Mailed check
7039      Credit card (automatic)
7040             Electronic check
7041                 Mailed check
7042    Bank transfer (automatic)
Name: PaymentMethod, Length: 7043, dtype: object


In [17]:
freq_map = X_train['PaymentMethod'].value_counts(normalize=True)
X_train_freq = X_train.copy()
X_train_freq['PaymentMethod_freq'] = X_train_freq['PaymentMethod'].map(freq_map)
print(X_train_freq[['PaymentMethod', 'PaymentMethod_freq']].drop_duplicates().sort_values('PaymentMethod_freq', ascending=False))

                  PaymentMethod  PaymentMethod_freq
3738           Electronic check            0.335641
3151               Mailed check            0.228257
4559  Bank transfer (automatic)            0.220802
3867    Credit card (automatic)            0.215300


### Task 1.7 (Challenge) — Target Encode `PaymentMethod`
Compute the mean of `y_train` grouped by `PaymentMethod`, using
`X_train`/`y_train` ONLY. Create `PaymentMethod_target` on a copy of
`X_train`. Then explain in a comment: what would go wrong if you
computed this mean using the FULL dataset (train + test) instead?


In [18]:
train_demo = X_train.copy()
train_demo['Churn_encoded'] = y_train.values
target_means = train_demo.groupby('PaymentMethod')['Churn_encoded'].mean()

X_train_target = X_train.copy()
X_train_target['PaymentMethod_target'] = X_train_target['PaymentMethod'].map(target_means)
print(target_means.sort_values(ascending=False))
# Using the full dataset would let real churn outcomes from the test set leak into this feature, making test accuracy look better than it really is.

PaymentMethod
Electronic check             0.457430
Mailed check                 0.192846
Bank transfer (automatic)    0.161576
Credit card (automatic)      0.149217
Name: Churn_encoded, dtype: float64


## 2. ColumnTransformer — Brief

Real datasets mix numeric and categorical columns that need different
treatment. `ColumnTransformer` routes each group of columns to its own
transformer (e.g. `StandardScaler` for numeric, `OneHotEncoder` for
categorical) and concatenates the results into one feature matrix — in
a single object, safe to reuse identically on train, test, and
production data.


### Task 2.1 — Define your column groups
Create two Python lists:
- `numeric_features`: all numeric columns (`tenure`, `MonthlyCharges`, `TotalCharges`, `SeniorCitizen`)
- `categorical_features`: every remaining column in `X_train` that is NOT in `numeric_features`

Do not hardcode the categorical list by hand — derive it programmatically
from `X_train.columns`.


In [19]:
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
categorical_features = [c for c in X_train.columns if c not in numeric_features]
print(numeric_features)
print(categorical_features)

['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


### Task 2.2 — Build the ColumnTransformer
Build a `ColumnTransformer` named `preprocessor` with:
- a numeric branch: `SimpleImputer(strategy='median')` -> `StandardScaler()`
- a categorical branch: `SimpleImputer(strategy='most_frequent')` -> `OneHotEncoder(handle_unknown='ignore')`

Use `Pipeline` for each branch (two mini-pipelines inside the
ColumnTransformer — this is a very common real-world pattern).


In [20]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

### Task 2.3 — Fit and inspect
`fit_transform` your `preprocessor` on `X_train`. Print the shape before
and after transformation, and explain in one comment why the number of
columns changed.


In [21]:
X_train_transformed = preprocessor.fit_transform(X_train)
print(X_train.shape)
print(X_train_transformed.shape)
# The column count grows because OneHotEncoder turns each category into several 0/1 columns instead of just one column per feature.

(5634, 19)
(5634, 45)


### Self-check 2.3

In [22]:
try:
    assert X_train_transformed.shape[0] == X_train.shape[0]
    assert X_train_transformed.shape[1] > X_train.shape[1]
    print("Self-check passed.")
except NameError:
    print("preprocessor / X_train_transformed not defined yet — complete Task 2.2-2.3 first.")
except AssertionError:
    print("Shape looks wrong — the transformed matrix should have MORE columns than the raw one (because of One-Hot Encoding).")

Self-check passed.


### Task 2.4 (Challenge) — `remainder` parameter
Rebuild the same `ColumnTransformer` but this time deliberately leave
ONE categorical column out of `categorical_features` when you define
the transformer list. Run it once with `remainder='drop'` and once with
`remainder='passthrough'`. Print the resulting shapes for both and
explain the difference in one comment.


In [23]:
categorical_features_missing_one = categorical_features[:-1]  # leave the last categorical column out on purpose

preprocessor_drop = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features_missing_one)
], remainder='drop')

preprocessor_passthrough = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features_missing_one)
], remainder='passthrough')

print("drop shape:", preprocessor_drop.fit_transform(X_train).shape)
print("passthrough shape:", preprocessor_passthrough.fit_transform(X_train).shape)
# 'drop' throws the leftover column away entirely. 'passthrough' keeps it, unencoded, as an extra column at the end.

drop shape: (5634, 41)
passthrough shape: (5634, 42)


## 3. Pipeline — Brief

A `Pipeline` chains preprocessing steps and a final estimator into ONE
object with a single `.fit()` / `.predict()`. This prevents data
leakage, makes cross-validation correct, and is what you'd actually
ship to production.


### Task 3.1 — Build a full Pipeline
Combine your `preprocessor` (from Task 2.2, using `remainder='drop'`
version) with a `LogisticRegression(max_iter=1000, random_state=42)`
into a `Pipeline` named `log_reg_pipeline`.


In [24]:
log_reg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

### Task 3.2 — Fit, predict, evaluate
Fit `log_reg_pipeline` on the training data, predict on the test data,
and print the accuracy and full `classification_report`.


In [25]:
log_reg_pipeline.fit(X_train, y_train)
y_pred = log_reg_pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.8055358410220014
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409



### Self-check 3.2

In [26]:
try:
    acc = accuracy_score(y_test, log_reg_pipeline.predict(X_test))
    assert acc > 0.70
    print(f"Self-check passed. Accuracy = {acc:.4f}")
except NameError:
    print("log_reg_pipeline not fitted yet — complete Task 3.1-3.2 first.")
except AssertionError:
    print(f"Accuracy ({acc:.4f}) looks too low — double check your preprocessing and target.")

Self-check passed. Accuracy = 0.8055


### Task 3.3 — Swap the model
Copy your pipeline and swap only the final estimator for a
`RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42)`.
You should only need to change ONE line. Fit and print its test accuracy.


In [27]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42))
])

rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)
print("Random Forest accuracy:", accuracy_score(y_test, rf_pred))

Random Forest accuracy: 0.8026969481902059


### Task 3.4 (Challenge) — GridSearchCV
Use `GridSearchCV` on `log_reg_pipeline` to search over at least 4
different values of `classifier__C`, with `cv=5`. Print the best
parameters and the best cross-validated accuracy.


In [28]:
param_grid = {'classifier__C': [0.01, 0.1, 1, 10]}

grid_search = GridSearchCV(log_reg_pipeline, param_grid, cv=5)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best CV accuracy:", grid_search.best_score_)

Best params: {'classifier__C': 10}
Best CV accuracy: 0.8045768249380222


## 4. make_pipeline vs Pipeline — Brief

`make_pipeline()` builds the exact same kind of object as `Pipeline()`,
but auto-generates step names from each object's lowercase class name
instead of letting you choose them. Behavior is identical; only the
step names (and therefore `GridSearchCV` parameter names) differ.


### Task 4.1 — Rebuild with make_pipeline
Rebuild the exact same Logistic Regression pipeline from Task 3.1, this
time using `make_pipeline()` instead of `Pipeline()`. Call it `pipe_mp`.


In [29]:
pipe_mp = make_pipeline(preprocessor, LogisticRegression(max_iter=1000, random_state=42))

### Task 4.2 — Compare step names
Fit `pipe_mp` on the training data. Print `.named_steps.keys()` for
BOTH `log_reg_pipeline` and `pipe_mp` side by side.


In [30]:
pipe_mp.fit(X_train, y_train)

print("Pipeline steps:", list(log_reg_pipeline.named_steps.keys()))
print("make_pipeline steps:", list(pipe_mp.named_steps.keys()))

Pipeline steps: ['preprocessor', 'classifier']
make_pipeline steps: ['columntransformer', 'logisticregression']


### Task 4.3 — Verify identical predictions
Prove that `log_reg_pipeline` and `pipe_mp` produce IDENTICAL
predictions on `X_test` using `np.array_equal`.


In [31]:
same = np.array_equal(log_reg_pipeline.predict(X_test), pipe_mp.predict(X_test))
print("Identical predictions?", same)

Identical predictions? True


### Self-check 4.3

In [32]:
try:
    same = np.array_equal(log_reg_pipeline.predict(X_test), pipe_mp.predict(X_test))
    assert same
    print("Self-check passed — predictions are identical, as expected.")
except NameError:
    print("pipe_mp not fitted yet — complete Task 4.1-4.2 first.")
except AssertionError:
    print("Predictions differ — something in pipe_mp is not built the same way as log_reg_pipeline.")

Self-check passed — predictions are identical, as expected.


### Task 4.4 (Challenge) — GridSearchCV with make_pipeline
Repeat Task 3.4's grid search, but this time on `pipe_mp`. You will need
to figure out the correct auto-generated parameter name yourself
(hint: print `pipe_mp.get_params().keys()` and look for the one ending
in `__C`).


In [33]:
print([k for k in pipe_mp.get_params().keys() if k.endswith('__C')])

param_grid_mp = {'logisticregression__C': [0.01, 0.1, 1, 10]}

grid_search_mp = GridSearchCV(pipe_mp, param_grid_mp, cv=5)
grid_search_mp.fit(X_train, y_train)

print("Best params:", grid_search_mp.best_params_)
print("Best CV accuracy:", grid_search_mp.best_score_)

['logisticregression__C']
Best params: {'logisticregression__C': 10}
Best CV accuracy: 0.8045768249380222


## 5. Final Mini-Project (Challenge)

Put everything together. No more step-by-step hand-holding.

**Your task:**
1. Build the best pipeline you can for predicting `Churn` on this
   dataset (you choose: Logistic Regression, Random Forest, or anything
   else from sklearn you're comfortable with).
2. Tune at least one hyperparameter with `GridSearchCV` or
   `cross_val_score`.
3. Report final **test set** accuracy AND a full `classification_report`.
4. Save your final trained pipeline to a file called
   `my_churn_pipeline.pkl` using `joblib`.
5. In a markdown cell below your code, write 3-4 sentences explaining:
   - Which encoding choices you made and why.
   - Whether `ColumnTransformer` + `Pipeline` made this easier than doing
     it manually, and specifically how.


In [34]:
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

param_grid_final = {'classifier__C': [0.01, 0.1, 1, 10, 100]}
final_search = GridSearchCV(final_pipeline, param_grid_final, cv=5, scoring='accuracy')
final_search.fit(X_train, y_train)

best_pipeline = final_search.best_estimator_
final_pred = best_pipeline.predict(X_test)

print("Best params:", final_search.best_params_)
print("Test accuracy:", accuracy_score(y_test, final_pred))
print(classification_report(y_test, final_pred))

import joblib
joblib.dump(best_pipeline, 'my_churn_pipeline.pkl')
print("Saved: my_churn_pipeline.pkl")

Best params: {'classifier__C': 100}
Test accuracy: 0.8005677785663591
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.65      0.55      0.59       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.73      1409
weighted avg       0.79      0.80      0.80      1409

Saved: my_churn_pipeline.pkl


### Your write-up

I used One-Hot Encoding for nominal columns like gender and InternetService since they have no real order, ordinal encoding for Contract because month-to-month, one year, and two year have a clear ranking, and left numeric columns to be imputed and scaled. ColumnTransformer and Pipeline made this much easier than doing it manually: instead of writing separate encoding code and carefully repeating it the same way on test data, one .fit() and one .predict() handled preprocessing and modeling together, with no risk of forgetting a step or leaking test information into training.

---
### Done?
Go back to `Encoding_Pipelines_Guide.docx` Section 5 (Cheat Sheet) and
`Telco_Churn_Encoding_Pipelines.ipynb` and compare your answers. If
something doesn't match and you don't understand why — that's the part
you actually need to study, not the part you got right.
